In [1]:
import time
import requests

BASE_URL = "http://localhost:8000"
API_KEY = "token-abc123"
HEADERS = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json",
}


def chat_once(prompt, model, temperature=0.0, top_p=1.0, top_k=None, max_tokens=128):
    payload = {
        "model": model,
        "messages": [{"role": "user", "content": prompt}],
        "temperature": temperature,
        "top_p": top_p,
        "max_tokens": max_tokens,
    }
    if top_k is not None:
        payload["top_k"] = top_k

    t0 = time.perf_counter()
    r = requests.post(
        f"{BASE_URL}/v1/chat/completions",
        headers=HEADERS,
        json=payload,
        timeout=120,
    )
    r.raise_for_status()
    data = r.json()
    dt = time.perf_counter() - t0

    return {
        "text": data["choices"][0]["message"]["content"],
        "usage": data.get("usage", {}),
        "latency_s": dt,
        "raw": data,
    }

/home/ubuntu/miniconda3/envs/llm/lib/python3.12/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [3]:
def main():
    # --- 测试配置 ---
    # 请确保这里填写的是你本地模型服务中实际存在的模型名称
    MODEL_NAME = "Qwen3-0.6B" 
    
    # 你想要问的问题
    TEST_PROMPT = "你好，你是谁？"

    print(f"正在向 {BASE_URL} 发送请求...")
    result = chat_once(prompt=TEST_PROMPT, model=MODEL_NAME)

    if "error" in result:
        print(f"❌ 请求失败: {result['error']}")
    else:
        print(f"✅ 回复内容: {result['text']}")
        print(f"⏱️  本次请求耗时: {result['latency_s']:.2f} 秒")

if __name__ == "__main__":
    main()


正在向 http://localhost:8000 发送请求...


HTTPError: 404 Client Error: Not Found for url: http://localhost:8000/v1/chat/completions

In [4]:
import requests
import json

# 设置请求的 URL (如果在其他机器上运行，请将 localhost 替换为服务器的实际 IP)
url = "http://localhost:8000/v1/models"

# 设置请求头，vLLM 的 --api-key 对应 Bearer Token 认证
headers = {
    "Authorization": "Bearer token-abc123"
}

try:
    # 发送 GET 请求
    response = requests.get(url, headers=headers)
    
    # 检查响应状态码
    if response.status_code == 200:
        print("✅ 成功获取模型列表！\n")
        # 格式化输出 JSON 数据
        data = response.json()
        print(json.dumps(data, indent=4, ensure_ascii=False))
    else:
        print(f"❌ 请求失败，状态码: {response.status_code}")
        print("错误信息:", response.text)

except requests.exceptions.RequestException as e:
    print(f"请求发生异常: {e}")

✅ 成功获取模型列表！

{
    "object": "list",
    "data": [
        {
            "id": "/home/ubuntu/models/Qwen3-0.6B",
            "object": "model",
            "created": 1774604650,
            "owned_by": "vllm",
            "root": "/home/ubuntu/models/Qwen3-0.6B",
            "parent": null,
            "max_model_len": 40960,
            "permission": [
                {
                    "id": "modelperm-93caadd28978536c",
                    "object": "model_permission",
                    "created": 1774604650,
                    "allow_create_engine": false,
                    "allow_sampling": true,
                    "allow_logprobs": true,
                    "allow_search_indices": false,
                    "allow_view": true,
                    "allow_fine_tuning": false,
                    "organization": "*",
                    "group": null,
                    "is_blocking": false
                }
            ]
        }
    ]
}


In [5]:
import requests
import json

# 设置请求的 URL，注意这里改为了 chat/completions 接口
url = "http://localhost:8000/v1/chat/completions"

# 设置请求头，必须包含 Content-Type 和 Authorization
headers = {
    "Content-Type": "application/json",
    "Authorization": "Bearer token-abc123"
}

# 构建请求体
payload = {
    # 这里的模型名称需要和您启动 vLLM 时的路径（或 /v1/models 返回的 id）保持一致
    "model": "/home/ubuntu/models/Qwen3-0.6B", 
    "messages": [
        {"role": "system", "content": "你是一个乐于助人的AI助手。"},
        {"role": "user", "content": "你好！请用简单的一句话介绍一下你自己。"}
    ],
    "temperature": 0.7,  # 控制生成的随机性，值越大越有创造性 (0.0 - 2.0)
    "max_tokens": 512    # 限制模型返回的最大 token 数量
}

print("⏳ 正在向 Qwen 模型发送请求，请稍候...\n")

try:
    # 发送 POST 请求
    response = requests.post(url, headers=headers, json=payload)
    
    # 检查响应状态码
    if response.status_code == 200:
        # 解析返回的 JSON 数据
        result = response.json()
        
        # 提取模型回复的具体文本内容
        assistant_message = result["choices"][0]["message"]["content"]
        
        print("🤖 模型回复:")
        print("-" * 30)
        print(assistant_message)
        print("-" * 30)
        
        # 如果你想查看完整的消耗信息，可以打印 usage
        # print(f"\n[Token 消耗统计]: {result['usage']}")
        
    else:
        print(f"❌ 请求失败，状态码: {response.status_code}")
        print("错误信息:", response.text)

except requests.exceptions.RequestException as e:
    print(f"请求发生异常: {e}")

⏳ 正在向 Qwen 模型发送请求，请稍候...

🤖 模型回复:
------------------------------
<think>
好的，用户让我用简单的一句话介绍一下自己。首先，我需要确保回答准确且简洁。作为AI助手，我应该明确说明我的功能和能力。用户可能希望快速了解我的用途，所以需要简明扼要。

接下来，我要考虑用户的潜在需求。他们可能是在测试我的反应，或者想了解我的角色。所以回答中需要涵盖几个关键点：我是AI助手，提供帮助，以及我的主要功能。同时，要保持口语化，避免使用复杂词汇。

然后，检查是否有需要补充的信息。比如，是否需要提到我的局限性？不过用户只要求一句话，所以可能不需要。保持回答自然，避免冗长。

最后，确保语句流畅，信息完整，符合用户的要求。确认没有语法错误，并且符合中文表达习惯。这样用户就能得到一个清晰、简洁的自我介绍。
</think>

我是AI助手，能帮助你解答问题或提供支持。
------------------------------
